
# Delivery Intelligence
## Análise exploratória e desenvolvimento do modelo de previsão de atrasos

### 1. Contextualização

O Delivery Intelligence é um projeto de Machine Learning desenvolvido a partir do Brazilian E-Commerce Public Dataset by Olist.

O objetivo é investigar padrões associados aos atrasos nas entregas de pedidos de e-commerce e desenvolver um modelo capaz de estimar o risco de atraso utilizando informações disponíveis no momento da compra.

### Objetivos

- Explorar e compreender os dados de pedidos.
- Investigar características associadas aos atrasos.
- Preparar os dados para treinamento.
- Desenvolver e avaliar modelos de classificação.
- Disponibilizar o modelo por meio de uma API REST com FastAPI.

### Definição do problema

Um pedido será considerado atrasado quando a data efetiva de entrega for posterior à data estimada.

- **1:** pedido atrasado.
- **0:** pedido não atrasado.

A análise considera pedidos entregues com data efetiva de entrega conhecida.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid")

# Caminho dos dados
DATA_DIR = Path("../data/raw")

print("Bibliotecas carregadas com sucesso!")
print(f"Diretório dos dados: {DATA_DIR.resolve()}")


Bibliotecas carregadas com sucesso!
Diretório dos dados: C:\Users\nanda\ProjetosIA\delivery-intelligence\data\raw


In [2]:
# Carregamento dos principais datasets

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

print("Arquivos carregados com sucesso!")

Arquivos carregados com sucesso!


In [3]:
# Dimensões dos datasets

print("Pedidos:", orders.shape)
print("Itens:", order_items.shape)
print("Clientes:", customers.shape)
print("Vendedores:", sellers.shape)

Pedidos: (99441, 8)
Itens: (112650, 7)
Clientes: (99441, 5)
Vendedores: (3095, 4)


In [4]:
print("ESTRUTURA DOS PEDIDOS")
orders.info()

print("\nESTRUTURA DOS ITENS")
order_items.info()

print("\nESTRUTURA DOS CLIENTES")
customers.info()

print("\nESTRUTURA DOS VENDEDORES")
sellers.info()

ESTRUTURA DOS PEDIDOS
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

ESTRUTURA DOS ITENS
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  

In [5]:
display(orders.head())
display(order_items.head())
display(customers.head())
display(sellers.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [6]:
display(orders.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
display(order_items.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
display(customers.head())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
display(sellers.head())

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



## 2. Preparação e integração dos dados

Nesta etapa, serão realizadas as transformações necessárias para construir uma base consolidada por pedido.

As tabelas de pedidos, clientes, itens e vendedores serão integradas por meio de seus identificadores.

Também serão convertidas as colunas temporais e tratados os registros que não possuem informações suficientes para identificar o resultado da entrega.


In [10]:

# 2.1 Conversão das colunas de data

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# 2.2 Seleção dos pedidos entregues com data efetiva conhecida

orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

# 2.3 Criação da variável-alvo

orders["is_late"] = (
    orders["order_delivered_customer_date"] >
    orders["order_estimated_delivery_date"]
).astype(int)

print("Pedidos válidos:", len(orders))
print("\nDistribuição da variável-alvo:")
print(orders["is_late"].value_counts())
print("\nProporção de atraso:")
print(orders["is_late"].value_counts(normalize=True).mul(100).round(2))


Pedidos válidos: 96470

Distribuição da variável-alvo:
is_late
0    88644
1     7826
Name: count, dtype: int64

Proporção de atraso:
is_late
0   91.89
1    8.11
Name: proportion, dtype: float64



## 3. Engenharia de atributos

Nesta etapa, as informações das diferentes tabelas serão agregadas por pedido.

Serão criadas variáveis relacionadas às características do pedido, aos vendedores envolvidos, à localização do cliente e ao período da compra.

O objetivo é representar cada pedido por um conjunto de atributos disponíveis no momento do checkout.


In [12]:

# 3.1 Informações do cliente
base = orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

# 3.2 Informações dos itens e vendedores
items_sellers = order_items.merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

# 3.3 Agregações dos itens por pedido
features_items = items_sellers.groupby("order_id").agg(
    quantidade_itens=("order_item_id", "count"),
    valor_produtos=("price", "sum"),
    valor_frete=("freight_value", "sum"),
    quantidade_vendedores=("seller_id", "nunique"),
    quantidade_estados_vendedores=("seller_state", "nunique")
).reset_index()

# 3.4 Identificação de vendedores no mesmo estado do cliente
items_sellers = items_sellers.merge(
    base[["order_id", "customer_state"]],
    on="order_id",
    how="inner"
)

items_sellers["mesmo_estado"] = (
    items_sellers["seller_state"] == items_sellers["customer_state"]
)

features_estado = items_sellers.groupby("order_id").agg(
    tem_vendedor_mesmo_estado=("mesmo_estado", "max")
).reset_index()

# 3.5 Integração das agregações
base = base.merge(features_items, on="order_id", how="inner")
base = base.merge(features_estado, on="order_id", how="left")

# 3.6 Variável de múltiplos vendedores
base["tem_multiplos_vendedores"] = (
    base["quantidade_vendedores"] > 1
).astype(int)

# 3.7 Atributos temporais
base["dia_semana_compra"] = (
    base["order_purchase_timestamp"].dt.day_of_week
)

base["hora_compra"] = (
    base["order_purchase_timestamp"].dt.hour
)

base["mes_compra"] = (
    base["order_purchase_timestamp"].dt.month
)

base["prazo_estimado_dias"] = (
    base["order_estimated_delivery_date"] -
    base["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# 3.8 Conversão para inteiros
base["tem_vendedor_mesmo_estado"] = (
    base["tem_vendedor_mesmo_estado"].astype(int)
)

print("Base consolidada:", base.shape)


Base consolidada: (96470, 21)


In [13]:
features = [
    "customer_state",
    "quantidade_itens",
    "valor_produtos",
    "valor_frete",
    "quantidade_vendedores",
    "quantidade_estados_vendedores",
    "tem_vendedor_mesmo_estado",
    "tem_multiplos_vendedores",
    "dia_semana_compra",
    "hora_compra",
    "mes_compra",
    "prazo_estimado_dias"
]

print("Quantidade de atributos:", len(features))

print("\nValores ausentes:")
display(base[features].isna().sum())

print("\nEstatísticas dos atributos numéricos:")
display(base[features].describe())

print("\nQuantidade de pedidos duplicados:")
print(base["order_id"].duplicated().sum())

Quantidade de atributos: 12

Valores ausentes:


customer_state                   0
quantidade_itens                 0
valor_produtos                   0
valor_frete                      0
quantidade_vendedores            0
quantidade_estados_vendedores    0
tem_vendedor_mesmo_estado        0
tem_multiplos_vendedores         0
dia_semana_compra                0
hora_compra                      0
mes_compra                       0
prazo_estimado_dias              0
dtype: int64


Estatísticas dos atributos numéricos:


,quantidade_itens,valor_produtos,valor_frete,quantidade_vendedores,quantidade_estados_vendedores,tem_vendedor_mesmo_estado,tem_multiplos_vendedores,dia_semana_compra,hora_compra,mes_compra,prazo_estimado_dias
count,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00,96470.00
mean,1.14,137.04,22.79,1.01,1.01,0.36,0.01,2.76,14.77,6.03,23.74
std,0.54,209.05,21.56,0.12,0.07,0.48,0.11,1.97,5.33,3.23,8.76
min,1.00,0.85,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,2.01
25%,1.00,45.90,13.85,1.00,1.00,0.00,0.00,1.00,11.00,3.00,18.33
50%,1.00,86.50,17.17,1.00,1.00,0.00,0.00,3.00,15.00,6.00,23.23
75%,1.00,149.90,24.02,1.00,1.00,1.00,0.00,4.00,19.00,8.00,28.41
max,21.00,13440.00,1794.96,5.00,3.00,1.00,1.00,6.00,23.00,12.00,155.14



Quantidade de pedidos duplicados:
0


## 4. Separação dos dados

Os atributos utilizados no treinamento representam informações disponíveis no momento da compra.

A variável `is_late` será utilizada como alvo. Para evitar vazamento de dados, as datas reais de entrega não serão utilizadas como atributos preditivos.

A divisão será cronológica, utilizando os pedidos mais antigos para treinamento e os mais recentes para teste.

In [14]:
# Ordenação cronológica
base = base.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Separação entre atributos e variável-alvo
X = base[features].copy()
y = base["is_late"].copy()

# Divisão 80% treino e 20% teste
limite = int(len(base) * 0.8)

X_train = X.iloc[:limite]
X_test = X.iloc[limite:]

y_train = y.iloc[:limite]
y_test = y.iloc[limite:]

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nPeríodo de treinamento:")
print(base.iloc[0]["order_purchase_timestamp"])
print("até", base.iloc[limite - 1]["order_purchase_timestamp"])

print("\nPeríodo de teste:")
print(base.iloc[limite]["order_purchase_timestamp"])
print("até", base.iloc[-1]["order_purchase_timestamp"])

Treino: (77176, 12)
Teste: (19294, 12)

Período de treinamento:
2016-09-15 12:16:38
até 2018-05-26 18:16:57

Período de teste:
2018-05-26 18:18:03
até 2018-08-29 15:00:37


In [15]:
import joblib

modelo_salvo = joblib.load("../models/modelo_atraso.pkl")

print(modelo_salvo)

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(transformers=[('numericas', StandardScaler(),
                                                  ['quantidade_itens',
                                                   'valor_produtos',
                                                   'valor_frete',
                                                   'quantidade_vendedores',
                                                   'quantidade_estados_vendedores',
                                                   'tem_vendedor_mesmo_estado',
                                                   'tem_multiplos_vendedores',
                                                   'dia_semana_compra',
                                                   'hora_compra', 'mes_compra',
                                                   'prazo_estimado_dias']),
                                                 ('categoricas',
                                                  OneHotEncoder(

## 5. Treinamento do modelo

Foi utilizada a Regressão Logística, um algoritmo de classificação supervisionada.

O pré-processamento foi organizado em um Pipeline:
- Padronização dos atributos numéricos com StandardScaler.
- Codificação do estado do cliente com OneHotEncoder.
- Classificação com LogisticRegression.

O parâmetro `class_weight='balanced'` foi utilizado para lidar com o desequilíbrio entre pedidos atrasados e não atrasados.

In [16]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

# Treinamento
modelo_salvo.fit(X_train, y_train)

# Previsões
y_pred = modelo_salvo.predict(X_test)
y_prob = modelo_salvo.predict_proba(X_test)[:, 1]

# Avaliação
print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

Acurácia: 0.5515
ROC-AUC: 0.7211

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.98      0.54      0.69     18273
           1       0.09      0.84      0.17      1021

    accuracy                           0.55     19294
   macro avg       0.54      0.69      0.43     19294
weighted avg       0.94      0.55      0.67     19294


Matriz de confusão:
[[9782 8491]
 [ 163  858]]


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score

for limiar in [0.5, 0.6]:
    previsoes = (y_prob >= limiar).astype(int)

    print(f"\nLimiar: {limiar}")
    print("Precisão:", round(precision_score(y_test, previsoes), 4))
    print("Recall:", round(recall_score(y_test, previsoes), 4))
    print("F1-score:", round(f1_score(y_test, previsoes), 4))


Limiar: 0.5
Precisão: 0.0918
Recall: 0.8404
F1-score: 0.1655

Limiar: 0.6
Precisão: 0.1006
Recall: 0.5162
F1-score: 0.1685


## 6. Avaliação do limiar de classificação

Foram comparados os limiares de 0.5 e 0.6.

O limiar de 0.5 apresentou recall de 84,04%, identificando a maior parte dos pedidos atrasados, porém com baixa precisão (9,18%).

Com o limiar de 0.6, a precisão aumentou para 10,06%, enquanto o recall caiu para 51,62%.

Como o objetivo é identificar possíveis atrasos, o limiar de 0.5 será mantido como referência inicial. A escolha definitiva deverá considerar o impacto operacional dos falsos positivos.

In [18]:
# Coeficientes da Regressão Logística
preprocessador = modelo_salvo.named_steps["preprocessamento"]
regressao = modelo_salvo.named_steps["modelo"]

nomes_atributos = preprocessador.get_feature_names_out()

importancia = pd.DataFrame({
    "atributo": nomes_atributos,
    "coeficiente": regressao.coef_[0]
})

importancia["impacto"] = importancia["coeficiente"].abs()

importancia = importancia.sort_values(
    "impacto",
    ascending=False
)

display(importancia.head(15))

,atributo,coeficiente,impacto
32,categoricas__customer_state_RR,1.55,1.55
12,categoricas__customer_state_AL,1.20,1.20
28,categoricas__customer_state_PR,-1.06,1.06
36,categoricas__customer_state_SP,-1.04,1.04
21,categoricas__customer_state_MG,-0.96,0.96
24,categoricas__customer_state_PA,0.82,0.82
31,categoricas__customer_state_RO,-0.80,0.80
17,categoricas__customer_state_DF,-0.75,0.75
10,numericas__prazo_estimado_dias,-0.68,0.68
20,categoricas__customer_state_MA,0.62,0.62


In [19]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(modelo_salvo, MODEL_DIR / "modelo_atraso.pkl")

joblib.dump(
    {"limiar": 0.6},
    MODEL_DIR / "configuracao.pkl"
)

print("Modelo e configuração salvos com sucesso!")

Modelo e configuração salvos com sucesso!


In [20]:
modelo_verificado = joblib.load("../models/modelo_atraso.pkl")
configuracao = joblib.load("../models/configuracao.pkl")

print("Modelo:", type(modelo_verificado))
print("Configuração:", configuracao)

Modelo: <class 'sklearn.pipeline.Pipeline'>
Configuração: {'limiar': 0.6}
